# QC-C2-03: OpenQASM 3.0 Transpiler across Qiskit, Cirq, CUDA-Q & QSDK
PESU – C-DAC Quantum Hackathon 2026

**Single-file version:** upload only this notebook to Qniverse JupyterLab and run all cells. The first cell unpacks the embedded `qnvkit` package; the second installs target frameworks for verification.

In [ ]:
# Unpacks the embedded qnvkit package next to this notebook (pure Python, needs only numpy).
import base64, io, os, sys, zipfile
_PKG = """
...
FgEAAAA=
"""
_dir = os.path.abspath("qnvkit_pkg")
zipfile.ZipFile(io.BytesIO(base64.b64decode(_PKG.replace("\n", "")))).extractall(_dir)
# Always use THIS embedded copy: put it first and drop any older qnvkit already imported or on the path.
sys.path[:] = [_dir] + [p for p in sys.path if p != _dir]
for _m in [m for m in sys.modules if m == "qnvkit" or m.startswith("qnvkit.")]:
    del sys.modules[_m]
print("qnvkit unpacked to", _dir)

In [ ]:
# Cell: Install execution frameworks for cross-framework verification (Qiskit, Aer, Cirq, Matplotlib)
# Purges cache first and uses --no-cache-dir to prevent container disk exhaustion
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--quiet", "qiskit", "qiskit-aer", "cirq-core", "matplotlib", "pylatexenc"], check=False)

In [ ]:
# Check available target frameworks in this Python environment.
# Note: Code generation works for ALL targets (Qiskit, Cirq, CUDA-Q, QSDK).
# Both Qiskit and Cirq enable cross-framework execution and equivalence checking.
import importlib, importlib.util, sys
importlib.invalidate_caches()
print("Python", sys.version.split()[0])
for _m in ("qiskit", "qiskit_aer", "cirq", "cudaq", "Qniverse", "qniverse"):
    status = "AVAILABLE (simulation & verification active)" if importlib.util.find_spec(_m) is not None else "not installed (code generated & AST verified)"
    print("%-12s %s" % (_m, status))

In [ ]:
import os, sys, json
import qnvkit
from qnvkit.transpile.api import transpile
from qnvkit.transpile.verify import compare_counts, format_counts_table
from qnvkit.qasm.parser3 import parse_qasm3
from qnvkit.qasm.diagnostics import QASMError

print("qnvkit loaded from:", os.path.dirname(qnvkit.__file__))
print("qnvkit transpiler ready!")

## 1. Selectable OpenQASM 3.0 Programs (Custom Gates, Modifiers, Loops, Dynamic Feedforward)
Choose `CIRCUIT_CHOICE = 1, 2, or 3` in the cell below to test different advanced OpenQASM 3.0 features:
- **Circuit 1:** Entangled State with Custom Gate `bell`, Modifiers (`ctrl @ rz`, `inv @ s`), Loop, Mid-Circuit Measurement, and Dynamic Conditional Feedforward (`if (m == 1) x q[2]`).
- **Circuit 2:** Quantum Teleportation Protocol with Entangled Bell Pair and Mid-Circuit Feedforward Correction (`if (m1 == 1) x`, `if (m0 == 1) z`).
- **Circuit 3:** 3-Qubit Parameterized Entangled State with Custom Gate, Loops, and Multi-Modifier Chains.

In [ ]:
# ==============================================================================
# Select Test Circuit:
# Change CIRCUIT_CHOICE to 1, 2, or 3 to test different OpenQASM 3.0 programs:
#   1: Entangled State with Custom Gate, Modifiers, For-Loop and Conditional Feedforward
#   2: Quantum Teleportation Protocol with Feedforward Classical Correction
#   3: 3-Qubit Parameterized Entangled State with Custom Gate, Loop and Modifiers
# ==============================================================================
CIRCUIT_CHOICE = 1

CIRCUITS = {
    1: """OPENQASM 3.0;
include "stdgates.inc";

gate bell a, b { h a; cx a, b; }      // custom gate
const float theta = pi / 4;

qubit[3] q;
bit[1] m;
bit[2] c;

bell q[0], q[1];
for int i in [0:1] {
    ctrl @ rz(theta * (i + 1)) q[i], q[2];   // modifier -> crz
}
inv @ s q[2];                          // modifier -> sdg
m[0] = measure q[0];
if (m == 1) x q[2];                    // classical condition
c[0] = measure q[1];
c[1] = measure q[2];
""",

    2: """OPENQASM 3.0;
include "stdgates.inc";

// Circuit 2: Quantum Teleportation Protocol with Feedforward Classical Correction
qubit[3] q;
bit[1] m0;
bit[1] m1;
bit[1] c;

// Prepare state to teleport on q[0] (|1>)
x q[0];

// Create Bell pair between Alice (q[1]) and Bob (q[2])
h q[1];
cx q[1], q[2];

// Alice interacts her state with her half of the Bell pair
cx q[0], q[1];
h q[0];
m0[0] = measure q[0];
m1[0] = measure q[1];

// Bob applies feedforward conditional corrections
if (m1 == 1) x q[2];
if (m0 == 1) z q[2];

// Verify teleported state on q[2]
c[0] = measure q[2];
""",

    3: """OPENQASM 3.0;
include "stdgates.inc";

// Circuit 3: Parameterized Entangled State with Custom Gate, Loop and Modifiers
gate bell a, b { h a; cx a, b; }

qubit[3] q;
bit[3] c;

bell q[0], q[1];
cx q[1], q[2];

for int i in [0:2] {
    inv @ s q[i];
    s q[i];
}
ctrl @ rx(pi) q[0], q[1];

c[0] = measure q[0];
c[1] = measure q[1];
c[2] = measure q[2];
"""
}

if CIRCUIT_CHOICE not in CIRCUITS:
    raise ValueError(f"Invalid CIRCUIT_CHOICE={CIRCUIT_CHOICE}. Choose 1, 2, or 3.")

QASM3_SOURCE = CIRCUITS[CIRCUIT_CHOICE]
print(f"=== Selected OpenQASM 3.0 Circuit #{CIRCUIT_CHOICE} ===\n")
print(QASM3_SOURCE)

from qnvkit.viz import draw_circuit
print("\n--- Input OpenQASM 3.0 Circuit Diagram (qc.draw()) ---")
draw_circuit(QASM3_SOURCE)

## 2. Generated code, verified (executed) per target

In [ ]:
results = transpile(QASM3_SOURCE, targets=["qiskit", "cirq", "cudaq", "qniverse"], options={"verify": True})
for target, res in results.items():
    print("=" * 70)
    print(target.upper(), "|", res.report.target_version, "| status:", res.report.status)
    print("-" * 70)
    print(res.code)

## 2b. Run it: the original circuit next to every generated program
`compare_counts` executes the **original OpenQASM circuit** on qnvkit's own reference simulator (pure NumPy,
independent of every target, and able to follow mid-circuit measurement, reset and conditions), then executes
the **generated** Qiskit, Cirq, CUDA-Q and QSDK programs for the same number of shots.

- `exact p` is the exact outcome probability of the original circuit.
- Each column holds the counts of one run; `TVD` is its total variation distance to `exact p`
  (0 = identical distribution; a 1024-shot run of a correct program stays well below 0.05).
- A framework that is not installed is listed as `NOT RUN`, never as a match.

In [ ]:
comparison = compare_counts(QASM3_SOURCE, shots=1024)
print(format_counts_table(comparison))

## 3. Structured report (CUDA-Q shown)

In [ ]:
print(results["cudaq"].report.to_json(indent=2))

## 4. Equivalence table
What was executed, fidelity (static circuits) or total variation distance (dynamic circuits, as here),
and what was not run. A framework that is not installed is reported as *not run*, never as verified.

In [ ]:
ver = results["qiskit"].report.metrics["cross_framework_verification"]
print("mode:", ver.get("mode"))
print("%-9s %-9s %-11s %-10s %s" % ("target", "executed", "equivalent", "fid/tvd", "note"))
for name, info in ver["frameworks"].items():
    val = info.get("fidelity", info.get("tvd"))
    print("%-9s %-9s %-11s %-10s %s" % (
        name, info.get("executed"), info.get("equivalent", "-"),
        "-" if val is None else "%.4f" % val, info.get("note") or info.get("error") or ""))

## 4b. Static circuit: statevector fidelity, including the Qniverse QSDK route
For a circuit without mid-circuit measurement or conditions, each generated program is executed and its
statevector compared with the reference simulator. The QSDK program is built with the QSDK, exported with
`qc.qasm()` and compared the same way.

In [ ]:
import qnvkit.qsdk_adapter as adapter
print("QSDK in use:", adapter.qsdk_description())

STATIC = """OPENQASM 3.0;
include "stdgates.inc";
qubit[3] q;
bit[3] c;
h q[0];
cx q[0], q[1];
ctrl @ ry(pi / 3) q[1], q[2];
U(0.3, 0.2, 0.1) q[0];
swap q[0], q[2];
c = measure q;
"""
static_results = transpile(STATIC, targets=["qiskit", "cirq", "cudaq", "qniverse"], options={"verify": True})
ver = static_results["qiskit"].report.metrics["cross_framework_verification"]
print("mode:", ver.get("mode"))
print("%-9s %-9s %-11s %-10s %s" % ("target", "executed", "equivalent", "fid/tvd", "note"))
for name, info in ver["frameworks"].items():
    val = info.get("fidelity", info.get("tvd"))
    print("%-9s %-9s %-11s %-10s %s" % (
        name, info.get("executed"), info.get("equivalent", "-"),
        "-" if val is None else "%.6f" % val, info.get("note") or info.get("error") or ""))
print()
print(static_results["qniverse"].code)

ns_static = {}
exec(static_results["qniverse"].code, ns_static)
qsdk_static = ns_static["build_circuit"]()
print("\n--- Static Circuit Diagram (qsdk_static.draw()) ---")
qsdk_static.draw()
print("\n--- Static circuit: original vs. generated programs ---")
comparison_static = compare_counts(STATIC, shots=1024)
print(format_counts_table(comparison_static))

## 5. Invalid input: precise `line:column` diagnostics

In [ ]:
bad_programs = {
    "unknown gate": 'OPENQASM 3.0;\nqubit[2] q;\nfoo q[0];\n',
    "index out of range": 'OPENQASM 3.0;\nqubit[2] q;\nh q[5];\n',
}
for title, src in bad_programs.items():
    try:
        parse_qasm3(src)
    except QASMError as e:
        print("[%s] line %d, col %d" % (title, e.line, e.col))
        print(e)
        print()

## 6. An unsupported construct is reported, never emitted as broken code

In [ ]:
UNSUPPORTED = """OPENQASM 3.0;
qubit[2] q;
bit[2] c;
h q[0];
c[0] = measure q[0];
if (c[0] == 1) x q[1];
"""
res = transpile(UNSUPPORTED, targets=["qiskit", "qniverse"])["qniverse"]
print("status:", res.report.status)
print(json.dumps(res.report.unsupported_constructs, indent=2))
compile(res.code, "qniverse", "exec")
print("generated QSDK code is still valid Python and omits the unsupported operation")

## 7. Optional: the same circuit as a Qniverse job
With an API key, the QSDK program generated from the static circuit is submitted to a Qniverse simulator and its
result is printed next to the exact distribution of the original circuit. Without a key this cell is skipped.

In [ ]:
token = os.environ.get("QNIVERSE_API_KEY", "")   # <- your Qniverse API key (keep it out of git)

if token.strip():
    import importlib, pkgutil, time
    import Qniverse
    from Qniverse.backend import Backend

    def _find(name):
        """Locates a job function wherever this QSDK version keeps it."""
        for m in pkgutil.walk_packages(Qniverse.__path__, Qniverse.__name__ + "."):
            try:
                mod = importlib.import_module(m.name)
            except Exception:
                continue
            if callable(getattr(mod, name, None)):
                return getattr(mod, name)
        raise ImportError("%s not found in the QSDK" % name)

    run = _find("run"); fetch_job_result = _find("fetch_job_result"); job_status = _find("job_status")

    ns = {}
    exec(static_results["qniverse"].code, ns)          # the generated QSDK program
    qsdk_circuit = ns["build_circuit"]()

    backend = Backend()
    backend.processor("CPU")
    backend.hpc("QACC_Cluster")
    backend.simulator("qasm_simulator")

    job = run(qsdk_circuit, backend, 1024, token=token, job_name="c2static")
    t0 = time.time()
    while time.time() - t0 < 300:
        status = str(job_status(token, job)).upper()
        if "COMPLETED" in status or "FAILED" in status:
            break
        time.sleep(3)
    print("QSDK in use:", adapter.qsdk_description())
    print("job", job, status)
    print("Qniverse result       :", fetch_job_result(token, job))
    print("exact (original QASM) :", {k: round(v, 4) for k, v in comparison_static["exact"].items()})
else:
    print("No token set: skipped.")